In [11]:
#!pip install --upgrade typing-extensions --user
#!pip install --upgrade plotly dash --user

In [12]:
# 1. Force the typing fix before doing anything
import typing
import typing_extensions
typing_extensions.Generic = typing.Generic

# 2. Base network and connection setup
import os
from datetime import datetime
from pymongo import MongoClient

# Configure Host IP
hostip = "192.168.1.108"
client = MongoClient(hostip, 27017)
db = client.a2_db
violations = db.violations_daily_summary

# 3. Now try the imports—Dash will no longer crash on typing_extensions
import plotly
import plotly.graph_objects as go
import plotly.express as px

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

In [13]:
# Initialize the Dash App inside the Jupyter Notebook
app = dash.Dash(__name__)

# Define the HTML Web Layout
app.layout = html.Div([
    html.H1("AWAS Real-Time Traffic Violation Dashboard",
            style={'textAlign': 'center', 'fontFamily': 'sans-serif',
                   'color': '#e84910', 'padding': '20px'}),
    
    # Dropdown for interactivity (e.g., Filtering by something specific if needed)
    html.Div([
        html.Label("Select Visual Profile:"),
        dcc.Dropdown(
            id='traffic-filter-dropdown',
            options=[
                {'label': 'All Live Violations', 'value': 'SPEED'},
            ],
            value='SPEED',
            clearable=False
        )
    ], style={'width': '30%', 'padding': '10px'}),
    
    # The Plotly Chart Component
    dcc.Graph(id='live-traffic-graph'),
    
    # Background timer triggers every 2000 milliseconds to check MongoDB
    dcc.Interval(
        id='interval-component',
        interval=2*1000, 
        n_intervals=0
    )
], style={'backgroundColor': '#f4e0b7', 'padding': '10px'})

In [ ]:
# Connects the Timer to the Plotly Graph Rendering Loop
@app.callback(
    Output('live-traffic-graph', 'figure'),
    [Input('interval-component', 'n_intervals'),
     Input('traffic-filter-dropdown', 'value')]
)
def update_live_graph(n, selected_filter):
    # Fetch latest data stream from MongoDB summary collection
    cursor = violations.find().sort("timestamp_end", -1).limit(10)
    data = list(cursor)
    
    if not data:
        return px.bar(title="Waiting for Spark Streaming Data Ingestion...")
        
    # Extract schema fields
    car_plates = [doc['car_plate'] for doc in data]
    max_speeds = [doc['max_speed_recorded'] for doc in data]
    total_counts = [doc['total_violations_today'] for doc in data]
    
    # Create interactive Plotly figure
    fig = px.bar(
        x=car_plates, 
        y=max_speeds,
        color=total_counts, # Color bars by how many times they've offended today
        title="Top 10 Most Critical Speeding Violations Detected",
        labels={
            'x': 'Vehicle Registration Plate', 
            'y': 'Maximum Recorded Speed (km/h)',
            'color': 'Total Infractions'
        },
        color_continuous_scale=px.colors.sequential.OrRd
    )
    
    # Add a horizontal threshold line for speed limit visibility
    fig.add_hline(y=110, line_dash="dash", line_color="red", annotation_text="Speed Limit (110km/h)")
    fig.update_layout(xaxis_tickangle=-45)
    
    return fig

# Launch the operational Dash framework server to be accessed externally
if __name__ == '__main__':
    app.run(mode='external', host='0.0.0.0', port=8050)